# 06 · Results: tables, figures, demo and poster

**Question.** What did we find? Everything in one place, regenerated from the run folders.

**What it runs.** Re-analyses every finished run, freezes the analysis folders into `results/`, rebuilds the interactive demo and the poster, and bundles everything for download.

**Runtime:** CPU · ~5 min

Every command is resumable: if Colab disconnects, re-run the same cell. Finished episodes are skipped and
cached calls are replayed without new API calls.

In [ ]:
# Setup 1/2 - get the code: clones the repo (or unzips /content/outbreak.zip if you uploaded it) and installs it
import pathlib, subprocess
REPO_URL = "https://github.com/albertogomezsoteres/outbreak.git"
if not pathlib.Path("/content/outbreak").exists():
    if pathlib.Path("/content/outbreak.zip").exists():
        subprocess.run(["unzip", "-q", "/content/outbreak.zip", "-d", "/content"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/outbreak"], check=True)
%cd /content/outbreak
!pip -q install -e ".[api,poster]"

In [ ]:
# Setup 2/2 - Drive for results, local cache (restored from Drive), API key from Colab Secrets
from outbreak.colab import setup, save_cache, show_report, show_figure
setup(need_key=False)

In [ ]:
import glob, shutil, os
for f in sorted(glob.glob("outputs/results/exp_*_main/episodes.jsonl")):
    !outbreak analyze {f}
for f in sorted(glob.glob("outputs/results/exp_*_main/outbreaks.jsonl")):
    !outbreak analyze {f}
for d in sorted(glob.glob("outputs/results/exp_*_main/analysis")):
    name = d.split("/")[-2].replace("_main", "")
    dst = f"results/{name}"; shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(d, dst, ignore=shutil.ignore_patterns("episodes_flat.csv"))
    print("frozen ->", dst)

## Headline figures

In [ ]:
for fig in ["results/exp_001/fig_leaderboard.png", "results/exp_002/fig_trait_heatmap.png",
            "outputs/results/exp_003_main/whitebox/fig_probe_layers.png", "outputs/results/exp_003_main/whitebox/fig_steering.png",
            "results/exp_004/fig_outbreak_growth.png", "results/exp_004/fig_frontier.png"]:
    show_figure(fig)

## Demo and poster

In [ ]:
!outbreak export-demo
!outbreak poster

## Bundle for the repo
Download, unzip into the repo, review `poster/content.yaml` (replace each `[pending ...]`), rebuild the poster locally, commit and tag `results-v1`.

In [ ]:
!cp outputs/results/natural_notes/*.jsonl data/processed/ 2>/dev/null; cp -r outputs/results/exp_003_main/whitebox results/whitebox 2>/dev/null; true
!zip -qr /content/outbreak_results.zip results data/processed demo poster/build outputs/results -x "*.sqlite*" "*.npz" "*episodes.jsonl"
from google.colab import files; files.download("/content/outbreak_results.zip")